# Autoload

Defines `AdlsAuth` and `setup_logger` inline (no filesystem copy needed).
Run from any notebook with: `%run ./autoload`


In [ ]:
# ── autoload.ipynb ───────────────────────────────────────────────────────────
# Inline definitions — no filesystem copy needed.
# Source of truth: databricks/lib/bobydo/adls_auth.py
# ─────────────────────────────────────────────────────────────────────────────

import logging
import sys
import types


def setup_logger(name: str = "de_project") -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:
        handler = logging.StreamHandler(sys.stdout)
        handler.setFormatter(logging.Formatter(
            fmt="%(asctime)s [%(levelname)-8s] %(name)s — %(message)s",
            datefmt="%Y-%m-%d %H:%M:%S"
        ))
        logger.addHandler(handler)
    logger.setLevel(logging.INFO)
    return logger


class AdlsAuth:
    REQUIRED_SECRETS = ["sp-client-id", "sp-client-secret", "sp-tenant-id"]

    def __init__(self, dbutils, spark, scope: str = "kv-scope"):
        self._dbutils = dbutils
        self._spark   = spark
        self._scope   = scope
        self._log     = setup_logger("AdlsAuth")

    def verify_secrets(self) -> None:
        self._log.info(f"Verifying secret scope: '{self._scope}'")
        try:
            scopes = [s.name for s in self._dbutils.secrets.listScopes()]
        except Exception as e:
            raise RuntimeError(f"Cannot list secret scopes: {e}") from e
        if self._scope not in scopes:
            raise RuntimeError(
                f"Secret scope '{self._scope}' not found. Available: {scopes}"
            )
        self._log.info(f"Scope '{self._scope}' found ✅")
        missing = []
        for key in self.REQUIRED_SECRETS:
            try:
                val = self._dbutils.secrets.get(scope=self._scope, key=key)
                self._log.info(f"  Secret '{key}' readable ({len(val)} chars) ✅")
            except Exception as e:
                self._log.error(f"  Secret '{key}' NOT readable: {e}")
                missing.append(key)
        if missing:
            raise RuntimeError(
                f"Missing secrets in scope '{self._scope}': {missing}"
            )
        self._log.info("All secrets verified ✅")

    def configure_oauth2(self, storage_account: str) -> None:
        self._log.info(f"Configuring OAuth2 for: {storage_account}")
        acct = f"{storage_account}.dfs.core.windows.net"
        try:
            client_id     = self._dbutils.secrets.get(self._scope, "sp-client-id")
            client_secret = self._dbutils.secrets.get(self._scope, "sp-client-secret")
            tenant_id     = self._dbutils.secrets.get(self._scope, "sp-tenant-id")
        except Exception as e:
            raise RuntimeError(f"Failed to read SP credentials: {e}") from e
        endpoint = f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
        try:
            self._spark.conf.set(f"fs.azure.account.auth.type.{acct}", "OAuth")
            self._spark.conf.set(
                f"fs.azure.account.oauth.provider.type.{acct}",
                "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
            )
            self._spark.conf.set(f"fs.azure.account.oauth2.client.id.{acct}",       client_id)
            self._spark.conf.set(f"fs.azure.account.oauth2.client.secret.{acct}",   client_secret)
            self._spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{acct}", endpoint)
        except Exception as e:
            raise RuntimeError(f"Failed to set Spark OAuth2 config: {e}") from e
        self._log.info(f"OAuth2 ready for {acct} ✅")

    def get_paths(self, storage_account: str) -> dict:
        def path(container):
            return f"abfss://{container}@{storage_account}.dfs.core.windows.net"
        return {"BRONZE": path("bronze"), "SILVER": path("silver"), "GOLD": path("gold")}

    def setup(self, storage_account: str) -> dict:
        self.verify_secrets()
        self.configure_oauth2(storage_account)
        paths = self.get_paths(storage_account)
        self._log.info(f"Setup complete — BRONZE={paths['BRONZE']}")
        return paths


# ── bobydo namespace — use bobydo.AdlsAuth(...) in all notebooks ─────────────
bobydo = types.SimpleNamespace(AdlsAuth=AdlsAuth, setup_logger=setup_logger)

print("✅ autoload ready — bobydo.AdlsAuth, bobydo.setup_logger")